<a href="https://colab.research.google.com/github/LathikaSakthivelCST/AI-Consumer-Rights-Assistant/blob/main/Consumer_Rights_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai gradio faiss-cpu sentence-transformers pypdf requests


In [2]:
import os
import requests
import gradio as gr
import faiss
import numpy as np

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from google.colab import userdata

print("All libraries imported successfully!")


All libraries imported successfully!


In [4]:
NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

if not NEXUS_API_KEY:
    raise ValueError("Nexus API key not found!")

client = OpenAI(
    api_key=NEXUS_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai/v1"
)

print("College Nexus API connected successfully!")


College Nexus API connected successfully!


In [7]:
models = client.models.list()

print("Available College Nexus Models:\n")

for model in models.data:
    print(model.id)


Available College Nexus Models:

nano-banana
whisper-1
nova-micro
text-embedding-3-small
gpt-4o-mini-tts
gpt-4.1-nano


In [8]:
response = client.chat.completions.create(
    model="nova-micro",
    messages=[
        {
            "role": "user",
            "content": "Explain the six consumer rights in India in simple English."
        }
    ]
)

print(response.choices[0].message.content)


Sure, here are the six consumer rights in India explained in simple English:

1. **Right to Safety**: 
   This means that you have the right to be protected against the marketing of goods and services that are harmful to your health and life. You should not be forced to use products that could cause you harm.

2. **Right to Information**:
   This right means that you have the right to get full and clear information about the product you are buying, like its price, quality, quantity, expiry date, manufacturer, and other important details. This helps you make an informed choice.

3. **Right to Choose**:
   This means you have the right to choose from a variety of products at competitive prices. You should not be forced to buy something you don't want.

4. **Right to Redress**:
   If you buy something and it doesn't work as it should, or if you face any unfair trade practice, you have the right to get it fixed, replaced, or to get a refund. You can also seek compensation if you’ve been ha

In [9]:
# STEP 6: CREATE CONSUMER RIGHTS KNOWLEDGE BASE

knowledge_base = [

    {
        "title": "Six Consumer Rights in India",
        "text": """
        Under the Consumer Protection Act 2019, India recognizes
        six consumer rights.

        Right to Safety: Protection from hazardous goods and services.

        Right to Information: Consumers have the right to know
        product quality, quantity, price, ingredients and standards.

        Right to Choose: Consumers can select from different
        products and services at competitive prices.

        Right to be Heard: Consumer complaints must receive
        proper consideration.

        Right to Seek Redressal: Consumers can seek solutions
        for unfair trade practices and consumer exploitation.

        Right to Consumer Education: Consumers have the right
        to learn about their rights and responsibilities.
        """
    },

    {
        "title": "Consumer Protection Act 2019",
        "text": """
        The Consumer Protection Act 2019 protects consumers
        in India.

        It covers offline and online purchases and services.

        It includes provisions related to e-commerce,
        misleading advertisements, product liability and
        unfair trade practices.

        It provides consumer dispute resolution through
        District, State and National Consumer Commissions.
        """
    },

    {
        "title": "Consumer Complaint and Remedies",
        "text": """
        Consumers can seek remedies when goods are defective
        or services are deficient.

        Possible remedies include removal of defects,
        replacement of goods, refund of the price,
        removal of service deficiencies and compensation
        for loss or injury, depending on the case.

        Consumers should preserve bills, receipts,
        order details, photographs and communication records.
        """
    },

    {
        "title": "Online Shopping Complaints",
        "text": """
        Consumers can raise complaints about online purchases.

        Common problems include non-delivery, defective products,
        wrong items, refund delays and misleading descriptions.

        Consumers should preserve the order ID, payment proof,
        seller details, delivery information and complaint messages.

        Contact the seller or platform and preserve the response.
        """
    },

    {
        "title": "National Consumer Helpline",
        "text": """
        The National Consumer Helpline is a Government of India
        consumer grievance support service.

        Consumers can contact the helpline at 1915.

        Official website:
        https://consumerhelpline.gov.in/

        The helpline provides channels to register
        consumer grievances.
        """
    }

]

print("Consumer Rights Knowledge Base created successfully!")

print("Total documents:", len(knowledge_base))

for doc in knowledge_base:
    print("-", doc["title"])


Consumer Rights Knowledge Base created successfully!
Total documents: 5
- Six Consumer Rights in India
- Consumer Protection Act 2019
- Consumer Complaint and Remedies
- Online Shopping Complaints
- National Consumer Helpline


In [10]:

# STEP 7: LOAD THE EMBEDDING MODEL

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [11]:

# STEP 8: TEST TEXT EMBEDDINGS

sample_text = [
    "I received a damaged mobile phone from an online shopping website."
]

sample_embedding = embedding_model.encode(sample_text)

print("Embedding generated successfully!")
print("Embedding shape:", sample_embedding.shape)


Embedding generated successfully!
Embedding shape: (1, 384)


In [12]:

# STEP 9: CREATE FAISS VECTOR DATABASE

import numpy as np
import faiss

# Extract document text
documents = [doc["text"] for doc in knowledge_base]

# Convert documents into embeddings
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

# Convert embeddings to float32
embeddings = embeddings.astype("float32")

# Get embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index
faiss_index = faiss.IndexFlatL2(dimension)

# Store embeddings
faiss_index.add(embeddings)

print("FAISS database created successfully!")
print("Total documents stored:", faiss_index.ntotal)
print("Embedding dimension:", dimension)


FAISS database created successfully!
Total documents stored: 5
Embedding dimension: 384


In [13]:

# STEP 10: TEST FAISS DATABASE

test_query = "I purchased a mobile phone online but received a damaged product."

# Convert complaint into embedding
query_embedding = embedding_model.encode(
    [test_query],
    convert_to_numpy=True
).astype("float32")

# Search FAISS database
distances, indices = faiss_index.search(
    query_embedding,
    3
)

print("User Complaint:", test_query)
print("\nMost Relevant Documents:\n")

for idx in indices[0]:
    print("Document:", knowledge_base[idx]["title"])
    print("-" * 50)


User Complaint: I purchased a mobile phone online but received a damaged product.

Most Relevant Documents:

Document: Online Shopping Complaints
--------------------------------------------------
Document: Consumer Complaint and Remedies
--------------------------------------------------
Document: Consumer Protection Act 2019
--------------------------------------------------


In [14]:

# STEP 11: DOCUMENT RETRIEVAL FUNCTION

def retrieve_documents(query, top_k=3):

    # Convert user complaint into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS database
    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    # Collect relevant documents
    results = []

    for idx in indices[0]:

        if idx != -1:
            results.append(knowledge_base[idx])

    return results


print("Document retrieval function created successfully!")


Document retrieval function created successfully!


In [15]:

# STEP 12: TEST DOCUMENT RETRIEVAL

test_complaint = "I ordered a mobile phone online but received a damaged product. I want a refund."

retrieved_docs = retrieve_documents(test_complaint)

print("USER COMPLAINT:")
print(test_complaint)

print("\nRETRIEVED DOCUMENTS:\n")

for doc in retrieved_docs:

    print("Title:", doc["title"])
    print("Information:", doc["text"][:250])
    print("-" * 60)


USER COMPLAINT:
I ordered a mobile phone online but received a damaged product. I want a refund.

RETRIEVED DOCUMENTS:

Title: Online Shopping Complaints
Information: 
        Consumers can raise complaints about online purchases.

        Common problems include non-delivery, defective products,
        wrong items, refund delays and misleading descriptions.

        Consumers should preserve the order ID, paymen
------------------------------------------------------------
Title: Consumer Complaint and Remedies
Information: 
        Consumers can seek remedies when goods are defective
        or services are deficient.

        Possible remedies include removal of defects,
        replacement of goods, refund of the price,
        removal of service deficiencies and com
------------------------------------------------------------
Title: Consumer Protection Act 2019
Information: 
        The Consumer Protection Act 2019 protects consumers
        in India.

        It covers offline an

In [16]:

# STEP 13: NEXUS AI COMPLAINT RESOLUTION ENGINE

def generate_complaint_response(user_complaint):

    # 1. Retrieve relevant documents
    retrieved_docs = retrieve_documents(user_complaint, top_k=3)

    # 2. Prepare context
    context = ""

    for doc in retrieved_docs:
        context += f"""
        Source: {doc['title']}
        Information: {doc['text']}
        """

    # 3. Prepare AI prompt
    prompt = f"""
    You are an AI Consumer Rights Assistant for India.

    Analyze the following consumer complaint.

    USER COMPLAINT:
    {user_complaint}

    RETRIEVED CONSUMER RIGHTS INFORMATION:
    {context}

    Provide your response using these sections:

    1. Complaint Understanding
    2. Complaint Category
    3. Relevant Consumer Rights
    4. Possible Solutions
    5. Step-by-Step Action Plan
    6. Evidence to Preserve
    7. Supporting Sources

    Instructions:
    - Use simple English.
    - Give practical suggestions.
    - Use the retrieved information as your knowledge source.
    - Do not invent legal sections or legal provisions.
    - If information is insufficient, clearly mention it.
    - Mention that this is general information, not legal advice.
    """

    # 4. Generate response using college Nexus API
    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful Indian Consumer Rights Assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    return response.choices[0].message.content


print("Nexus AI Complaint Resolution Engine created successfully!")


Nexus AI Complaint Resolution Engine created successfully!


In [17]:

# STEP 14: TEST COMPLAINT RESOLUTION ENGINE

complaint = """
I purchased a mobile phone from an online shopping website.
When the product arrived, the screen was broken.
The seller is refusing to replace the product or refund my money.
What should I do?
"""

answer = generate_complaint_response(complaint)

print("AI CONSUMER RIGHTS ASSISTANT")
print("=" * 60)
print(answer)


AI CONSUMER RIGHTS ASSISTANT
### 1. Complaint Understanding

You purchased a mobile phone from an online shopping website, but upon arrival, the screen was broken. The seller is not agreeing to replace the product or refund your money.

### 2. Complaint Category

This falls under the category of defective products in an online purchase.

### 3. Relevant Consumer Rights

Under the Consumer Protection Act 2019, you have the right to seek a replacement for a defective product or a refund if the seller refuses to replace it.

### 4. Possible Solutions

Possible solutions include:
- Replacement of the defective phone.
- Refund of the amount paid for the phone.

### 5. Step-by-Step Action Plan

1. **Document Everything:**
   - Take clear photographs of the broken screen.
   - Save all order details, including the order ID, payment proof, and seller details.
   - Keep all delivery information and any communication with the seller.

2. **Contact the Seller:**
   - Reach out to the seller again

In [ ]:

# STEP 15: BUILD GRADIO USER INTERFACE

def consumer_assistant(complaint):

    if not complaint or not complaint.strip():
        return "Please enter your consumer complaint."

    try:
        result = generate_complaint_response(complaint)
        return result

    except Exception as e:
        return f"Something went wrong: {str(e)}"


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Consumer Rights Assistant"
) as demo:

    gr.Markdown("""
    # 🛡️ AI-Powered Consumer Rights
    ## Complaint Resolution Assistant

    Understand your consumer rights and discover
    possible solutions to your complaints.

    Powered by Retrieval-Augmented Generation (RAG).
    """)

    gr.Markdown("### 📝 Describe Your Complaint")

    complaint_input = gr.Textbox(
        label="Your Complaint",
        placeholder="Example: I purchased a mobile phone online, but received a damaged product. The seller refuses to refund my money.",
        lines=5
    )

    submit_button = gr.Button(
        "🔍 Analyze My Complaint",
        variant="primary"
    )

    output = gr.Markdown(
        label="AI Complaint Resolution"
    )

    submit_button.click(
        fn=consumer_assistant,
        inputs=complaint_input,
        outputs=output
    )

    gr.Markdown("""
    ---
    **Disclaimer:** This assistant provides general
    consumer awareness information and is not a substitute
    for professional legal advice.

    **Technology:** Python | Nexus API | FAISS | RAG | Gradio
    """)

demo.launch(share=True, debug=True)


/tmp/ipykernel_2181/2630378062.py:16: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://405159284aa0879c85.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
